# Session API Reference

Complete coverage of every `TerrainSession` method and the server endpoint it calls.

| Group | Endpoints covered |
|---|---|
| Server | `GET /api/settings`, `GET /api/terrain/sources` |
| Regions | `GET /api/regions`, `POST`, `PUT`, `DELETE`, `GET/{name}/settings`, `PUT/{name}/settings` |
| Terrain | `POST /api/terrain/dem`, `GET /api/terrain/water-mask`, `GET /api/terrain/satellite` |
| Merge | `POST /api/dem/merge` |
| Cities | `GET /api/cities/cached`, `POST /api/cities`, `POST /api/cities/raster`, `POST /api/cities/export3mf` |
| Composite | `POST /api/composite/city-raster` |
| Export | `POST /api/export` (stl / obj / obj_split), `GET /api/export/obj/inspect`, `GET /api/export/obj/verify`, `POST /api/export/slice` |
| Cache | `GET /api/cache`, `DELETE /api/cache`, `GET /api/cache/check` |

---
## 0 — Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from terrain_session import TerrainSession

s = TerrainSession()   # default port 9000
s.start()              # launches uvicorn; kills any stale server on the same port

Killing stale server on port 9000 (PID 125104, C:\Python311\python.exe)
Server running (PID 57020, python: c:\Users\eac84\OneDrive\Documents\Projects\3D Maps\Code\.venv\Scripts\python.exe)


---
## 1 — Server settings  `GET /api/settings`

Returns everything the server considers authoritative: available projections,
DEM sources (with live `available` flags based on API keys / local files),
water datasets, slicer `.ini` files on disk, and valid numeric ranges.

In [2]:
cfg = s.server_settings()

# Inspect individual sections
import pandas as pd
display(pd.DataFrame(cfg["dem_sources"]))
display(pd.DataFrame(cfg["projections"]))
print("Slicer configs:", cfg["slicer_configs"])
print("DEM constraints:", cfg["dem_constraints"])

Projections  : ['none', 'cosine', 'mercator', 'equidistant', 'lambert', 'sinusoidal']
DEM sources  : ['local', 'h5_local', 'SRTMGL1', 'SRTMGL3', 'AW3D30', 'COP30', 'COP90', 'SRTM15Plus']
Water datasets: ['esa', 'jrc']
Slicer configs: ['maps_2025_part2.ini']


,id,name,description,resolution_m,requires_api_key,available
0,local,Local SRTM Tiles,30 m SRTM tiles from local disk cache,30,False,True
1,h5_local,Local SRTM H5 (90 m),High-fidelity SRTM3 from local strm_data.h5 — ...,90,False,True
2,SRTMGL1,SRTM 30m (Global),OpenTopography — 30 m resolution,30,True,False
3,SRTMGL3,SRTM 90m (Global),OpenTopography — 90 m resolution,90,True,False
4,AW3D30,ALOS World 3D 30m,OpenTopography — 30 m resolution,30,True,False
5,COP30,Copernicus DSM 30m,OpenTopography — 30 m resolution,30,True,False
6,COP90,Copernicus DSM 90m,OpenTopography — 90 m resolution,90,True,False
7,SRTM15Plus,SRTM15+ (Bathymetry+Land),OpenTopography — 500 m resolution,500,True,False


,id,name,description
0,none,None (Plate Carrée),No projection applied. Raw lat/lon grid. Fast ...
1,cosine,Cosine Correction,Simple horizontal scaling by cos(latitude). Re...
2,mercator,Web Mercator,Conformal cylindrical projection. Preserves sh...
3,equidistant,Equidistant Cylindrical,Simple projection preserving distances along m...
4,lambert,Lambert Equal-Area,Cylindrical equal-area projection. Preserves r...
5,sinusoidal,Sinusoidal,Pseudocylindrical equal-area. Good for single ...


Slicer configs: ['maps_2025_part2.ini']
DEM constraints: {'dim': {'min': 1, 'max': 2000, 'default': 800}, 'depth_scale': {'min': 0.0, 'max': 10.0, 'default': 0.5}, 'water_scale': {'min': 0.0, 'max': 1.0, 'default': 0.05}, 'sat_scale': {'min': 10, 'max': 5000, 'default': 500}}


### DEM sources endpoint  `GET /api/terrain/sources`

Lighter alternative — returns only DEM sources with availability flags.
Called directly here to show the raw response shape.

In [3]:
import requests
resp = requests.get("http://127.0.0.1:9000/api/terrain/sources")
resp.raise_for_status()
data = resp.json()
print("OpenTopo key configured:", data["opentopo_api_key_configured"])
print("H5 SRTM available:      ", data["h5_srtm_available"])
display(pd.DataFrame(data["sources"]))

OpenTopo key configured: False
H5 SRTM available:       True


,id,label,provider,resolution_m,requires_api_key,available,note
0,local,Local SRTM Tiles,local,30,False,True,NaN
1,h5_local,"Local SRTM H5 (City-scale, ~90m)",local_h5,90,False,True,High-fidelity SRTM3 from local strm_data.h5 — ...
2,SRTMGL1,SRTM 30m (Global),OpenTopography,30,True,False,NaN
3,SRTMGL3,SRTM 90m (Global),OpenTopography,90,True,False,NaN
4,AW3D30,ALOS World 3D 30m,OpenTopography,30,True,False,NaN
5,COP30,Copernicus DSM 30m,OpenTopography,30,True,False,NaN
6,COP90,Copernicus DSM 90m,OpenTopography,90,True,False,NaN
7,SRTM15Plus,SRTM15+ (Bathymetry+Land),OpenTopography,500,True,False,NaN


---
## 2 — Regions

### List  `GET /api/regions`

In [ ]:
# All regions
s.regions()

# Optional: filter by column
# s.regions(filter_col="continent", filter_val="North America")

### Create  `POST /api/regions`

In [ ]:
s.create_region(
    name="TestRegion",
    north=37.82, south=37.70,
    east=-122.35, west=-122.55,
    description="San Francisco Bay for testing",
    continent="North America",
    source="test",
)
# After create_region() the session is also selected on this region
print(s.region_name, s.bbox)

### Select  `GET /api/regions` + `GET /api/regions/{name}/settings`

In [7]:
# Switch to any saved region; loads bbox and merges saved settings over defaults
s.select("WestAmerica")
s.settings_table()   # shows all 8 groups: dem, export, split, slicer, water, satellite, city, view

Region : WestAmerica
BBox   : {'north': 49.2678, 'south': 30.44867, 'east': -101.60167, 'west': -126.91503}
── dem ──


,value
dim,800
depth_scale,0.5
water_scale,0.05
subtract_water,True
dem_source,local
projection,none
maintain_dimensions,False
clip_nans,False
show_sat,False


── export ──


,value
model_height,30.0
base_height,10.0
exaggeration,1.0
sea_level_cap,False
floor_val,0.0
engrave_label,False
label_text,
contours,False
contour_interval,100.0
contour_style,engraved


── split ──


...

,value
sat_scale,500
dataset,esa


── satellite ──


,value
dim,800


── city ──


TypeError: object of type 'float' has no len()

### Edit settings

Settings are grouped by the endpoint they feed.  Edit in-place before calling the corresponding method.

In [8]:
# ── DEM fetch ──────────────────────────────────────────────────────────────
# DEM sources (no API key): "local", "h5_local"
# DEM sources (need OPENTOPO_API_KEY): "SRTMGL1", "SRTMGL3", "AW3D30", "COP30", "COP90", "SRTM15Plus"
# Projections: "none", "cosine", "mercator", "equal_area", "equidistant", "lambert", "sinusoidal"
s.settings["dem"]["dim"]           = 800
s.settings["dem"]["depth_scale"]   = 0.5
s.settings["dem"]["water_scale"]   = 0.05
s.settings["dem"]["subtract_water"]= True
s.settings["dem"]["dem_source"]    = "local"
s.settings["dem"]["projection"]    = "none"
s.settings["dem"]["show_sat"]      = False   # include ESA/JRC land-cover overlay in DEM response

# ── Water mask / water overlay ─────────────────────────────────────────────
# dataset controls both /api/terrain/water-mask AND the dem show_sat overlay
s.settings["water"]["dataset"]     = "esa"   # "esa" | "jrc"
s.settings["water"]["sat_scale"]   = 500     # Earth Engine resolution in m/px (≥10)

# ── Satellite imagery ──────────────────────────────────────────────────────
s.settings["satellite"]["dim"]     = 800     # pixel resolution of returned JPEG (independent of dem.dim)

# ── Export ─────────────────────────────────────────────────────────────────
s.settings["export"]["model_height"]  = 30.0
s.settings["export"]["base_height"]   = 10.0
s.settings["export"]["exaggeration"]  = 1.0
s.settings["export"]["engrave_label"] = False
s.settings["export"]["contours"]      = False

# ── Puzzle split ───────────────────────────────────────────────────────────
s.settings["split"]["split_rows"]  = 2
s.settings["split"]["split_cols"]  = 2

# ── Slicer ─────────────────────────────────────────────────────────────────
s.settings["slicer"]["slicer_config"] = "maps_2025_part2.ini"
s.settings["slicer"]["output_subdir"] = "gcode"

# ── City / OSM ─────────────────────────────────────────────────────────────
s.settings["city"]["layers"]       = ["buildings", "roads", "waterways"]
s.settings["city"]["min_area"]     = 5.0     # m² — skip very small footprints

# ── Visualisation (local only — not sent to any endpoint) ─────────────────
s.settings["view"]["colormap"]     = "terrain"

s.settings_table()

── dem ──


,value
dim,800
depth_scale,0.5
water_scale,0.05
subtract_water,True
dem_source,local
projection,none
maintain_dimensions,False
clip_nans,False
show_sat,False


── export ──


,value
model_height,30.0
base_height,10.0
exaggeration,1.0
sea_level_cap,False
floor_val,0.0
engrave_label,False
label_text,
contours,False
contour_interval,100.0
contour_style,engraved


── split ──


...

,value
sat_scale,500
dataset,esa


── satellite ──


,value
dim,800


── city ──


TypeError: object of type 'float' has no len()

### Save settings  `PUT /api/regions/{name}/settings`

In [ ]:
# Persists DEM + view + water settings to the database so select() reloads them next time
s.save_settings()

### Update region  `PUT /api/regions/{name}`

In [ ]:
# Any argument left as None keeps the existing value
s.select("TestRegion")
s.update_region(description="Updated description", continent="North America")
print(s.bbox)

### Delete region  `DELETE /api/regions/{name}`

In [ ]:
# Removes the region from the database; clears selection if it matches current region
s.delete_region("TestRegion")
# Reselect a real region for the rest of the notebook
s.select("WestAmerica")

---
## 3 — Terrain

### DEM  `POST /api/terrain/dem`

In [9]:
s.select()
s.fetch_dem()
# Response stored on s.dem: {dem_values, dimensions, min_elevation, max_elevation, mean_elevation, bbox}

TypeError: TerrainSession.select() missing 1 required positional argument: 'name'

In [ ]:
s.show_dem()   # matplotlib figure — elevation coloured by settings["view"]["colormap"]

### Water mask  `GET /api/terrain/water-mask`

Returns a binary water mask plus the raw ESA WorldCover class array.
Configured via `settings["water"]`.

In [ ]:
s.fetch_water_mask()
wm = s.water_mask
print(f"Water pixels : {wm['water_pixels']} / {wm['total_pixels']}")
print(f"Water %      : {wm['water_percentage']:.2f}%")
print(f"Mask shape   : {wm['water_mask_dimensions']}")
print(f"ESA shape    : {wm['esa_dimensions']}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

H, W = wm["water_mask_dimensions"]
mask = np.array(wm["water_mask_values"]).reshape(H, W)
esa  = np.array(wm["esa_values"]).reshape(H, W)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(mask, cmap="Blues", origin="upper")
axes[0].set_title("Water mask (binary)")
axes[1].imshow(esa,  cmap="tab20", origin="upper")
axes[1].set_title("ESA WorldCover classes")
plt.tight_layout()
plt.show()

### Satellite imagery  `GET /api/terrain/satellite`

ESRI World Imagery WMTS tiles stitched to a single JPEG.
Configured via `settings["satellite"]`.

In [ ]:
s.fetch_satellite()
# s.satellite is a base64-encoded JPEG string
print(f"Satellite image: {len(s.satellite) // 1024} KB base64")

In [ ]:
import base64
from PIL import Image
from io import BytesIO

img = Image.open(BytesIO(base64.b64decode(s.satellite)))
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis("off")
plt.title(f"Satellite — {s.region_name}")
plt.show()

---
## 4 — DEM merge  `POST /api/dem/merge`

Composite multiple elevation layers into one DEM.  Each layer specifies a
source, optional per-layer processing, and a blend mode.

The result is stored on `s.dem` exactly like `fetch_dem()`,
so `show_dem()` and `export_obj()` work immediately after.

In [ ]:
# Minimal: one base layer
s.merge_dem(layers=[
    {"source": "local", "dim": 800, "blend_mode": "base", "weight": 1.0},
])
s.show_dem()

In [ ]:
# Advanced: blend two sources; apply smoothing to second layer
s.merge_dem(layers=[
    {
        "source": "local",
        "dim": 800,
        "blend_mode": "base",   # first layer is always the base
        "weight": 1.0,
        "processing": {
            "clip_min": 0.0,    # clamp ocean trenches to sea level
        },
    },
    {
        "source": "local",      # swap for "COP30" if you have an API key
        "dim": 400,
        "blend_mode": "blend",  # lerp weighted average into base
        "weight": 0.4,
        "processing": {
            "smooth_sigma": 1.5,  # Gaussian blur before blending
            "normalize": False,
        },
    },
])
s.show_dem()

In [ ]:
# River extraction: detect thin valleys and blend them into the base
s.merge_dem(layers=[
    {"source": "local", "dim": 800, "blend_mode": "base", "weight": 1.0},
    {
        "source": "local",
        "dim": 800,
        "blend_mode": "rivers",   # paints extracted river channels over base
        "weight": 1.0,
        "processing": {
            "extract_rivers": True,
            "river_max_width_px": 8,
        },
    },
])

---
## 5 — Cities / OSM

> The city endpoints require the region to be **≤ 15 km diagonal**.
> Switch to a small urban region before running these cells.

In [ ]:
# Select a small urban region — create a temporary one if needed
# Granada old town (~1.5 km)
s.create_region(
    name="GranadaOldTown",
    north=37.181, south=37.170,
    east=-3.589,  west=-3.601,
    continent="Europe",
)
s.select("GranadaOldTown")

### Cache check  `GET /api/cities/cached`

In [ ]:
resp = requests.get("http://127.0.0.1:9000/api/cities/cached", params={
    **s.bbox,
    "simplify_tolerance": s.settings["city"]["simplify_tolerance"],
    "min_area":           s.settings["city"]["min_area"],
})
resp.raise_for_status()
print(resp.json())

### Fetch OSM  `POST /api/cities`

In [ ]:
s.fetch_cities()
# s.city_data keys: buildings, roads, waterways (each a GeoJSON FeatureCollection)
buildings = s.city_data.get("buildings", {}).get("features", [])
print(f"Sample building: {buildings[0]['properties'] if buildings else 'none'}")

### City raster  `POST /api/cities/raster`

Burns GeoJSON onto a DEM-format height map (flat array).
Used by the CityRaster layer view — buildings raised, roads flat, waterways depressed.

In [ ]:
resp = requests.post("http://127.0.0.1:9000/api/cities/raster", json={
    **s.bbox,
    "dim":               s.settings["dem"]["dim"],
    "buildings":         s.city_data.get("buildings", {"type": "FeatureCollection", "features": []}),
    "roads":             s.city_data.get("roads",     {"type": "FeatureCollection", "features": []}),
    "waterways":         s.city_data.get("waterways", {"type": "FeatureCollection", "features": []}),
    "building_scale":    s.settings["city"]["building_scale"],
    "road_depression_m": s.settings["city"]["road_depression_m"],
    "water_depression_m":s.settings["city"]["water_depression_m"],
})
resp.raise_for_status()
raster = resp.json()
print(f"Raster size: {raster['width']} × {raster['height']}")
print(f"Height range: {raster['vmin']:.2f} – {raster['vmax']:.2f} m")

arr = np.array(raster["values"]).reshape(raster["height"], raster["width"])
plt.figure(figsize=(7, 7))
plt.imshow(arr, origin="upper", cmap="hot")
plt.colorbar(label="height (m)")
plt.title("City raster")
plt.show()

### Composite city raster  `POST /api/composite/city-raster`

Reads from OSM disk cache and returns **separate normalised arrays** per layer
(buildings, roads, waterways, walls).  ~50× faster than the raster endpoint
because it skips network I/O.  Weights are applied client-side.

Requires `fetch_cities()` to have been called first.

In [ ]:
s.composite_city_raster()   # width/height default to settings["dem"]["dim"]
cr = s.city_raster

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, layer in zip(axes, ["buildings", "roads", "waterways", "walls"]):
    arr = np.array(cr[layer]).reshape(cr["height"], cr["width"])
    ax.imshow(arr, origin="upper", cmap="hot")
    ax.set_title(layer)
    ax.axis("off")
plt.tight_layout()
plt.show()

### City 3MF export  `POST /api/cities/export3mf`

Generates a 3MF file with the terrain mesh plus extruded building prisms.
Requires DEM values from `s.dem` and buildings from `s.city_data`.

In [ ]:
# Fetch a low-res DEM first if not already done
s.settings["dem"]["dim"] = 200
s.fetch_dem()

resp = requests.post("http://127.0.0.1:9000/api/cities/export3mf", json={
    **s.bbox,
    "dem_values":      s.dem["dem_values"],
    "dem_width":       s.dem["dimensions"][1],
    "dem_height":      s.dem["dimensions"][0],
    "buildings":       s.city_data.get("buildings", {"type": "FeatureCollection", "features": []}),
    "model_height_mm": s.settings["export"]["model_height"],
    "base_mm":         s.settings["export"]["base_height"],
    "building_z_scale":s.settings["city"]["building_scale"],
    "simplify_terrain":s.settings["city"]["simplify_terrain"],
    "name":            s.region_name,
})
resp.raise_for_status()

out_path = f"../output/{s.region_name}_city.3mf"
with open(out_path, "wb") as f:
    f.write(resp.content)
print(f"Saved: {out_path}  ({len(resp.content) / 1024:.1f} KB)")

# Clean up temporary region
s.delete_region("GranadaOldTown")
s.select("WestAmerica")

---
## 6 — Export

All three formats go through `POST /api/export` with a `format` field.

### OBJ puzzle split  `POST /api/export` (format=obj_split)

In [ ]:
s.select("WestAmerica")
s.settings["dem"]["dim"]          = 800
s.settings["split"]["split_rows"] = 4
s.settings["split"]["split_cols"] = 4

s.export_obj()   # saves to strm2stl/output/; stores path on s.obj_path
print(s.obj_path)

### Single-piece OBJ  `POST /api/export` (format=obj)

In [ ]:
payload = {
    **s.bbox,
    "format": "obj",
    "name":   s.region_name,
    "dem":    s.settings["dem"],
    "export": s.settings["export"],
    "split":  s.settings["split"],
}
resp = requests.post("http://127.0.0.1:9000/api/export", json=payload, timeout=300)
resp.raise_for_status()

import pathlib
pathlib.Path("../output").mkdir(exist_ok=True)
out = pathlib.Path(f"../output/{s.region_name}_single.obj")
out.write_bytes(resp.content)
print(f"Saved: {out}  ({len(resp.content) / 1024:.1f} KB)")

### STL export  `POST /api/export` (format=stl)

In [ ]:
payload = {
    **s.bbox,
    "format": "stl",
    "name":   s.region_name,
    "dem":    s.settings["dem"],
    "export": s.settings["export"],
    "split":  s.settings["split"],
}
resp = requests.post("http://127.0.0.1:9000/api/export", json=payload, timeout=300)
resp.raise_for_status()

out = pathlib.Path(f"../output/{s.region_name}.stl")
out.write_bytes(resp.content)
print(f"Saved: {out}  ({len(resp.content) / 1024:.1f} KB)")

### Inspect OBJ  `GET /api/export/obj/inspect`

Fast parse — returns object names and piece counts without running mesh health checks.

In [ ]:
info = s.inspect_obj()
# {total, terrain_count, base_count, objects: [...]}

### Verify OBJ  `GET /api/export/obj/verify`

Full mesh health check: watertight, holes, non-manifold edges, z_min, volume.

In [ ]:
s.verify()

### Slice to gcode  `POST /api/export/slice`

Calls PrusaSlicer CLI on every terrain+base pair from the exported OBJ.
Configure via `settings["slicer"]`.

In [ ]:
s.settings["slicer"]["slicer_config"] = "maps_2025_part2.ini"
s.settings["slicer"]["output_subdir"] = "gcode"
result = s.slice()
print(result)

---
## 7 — Cache

### Status  `GET /api/cache`

In [ ]:
stats = s.cache_status()
display(pd.DataFrame(stats.get("cache_dirs", [])))

### Cache check  `GET /api/cache/check`

Returns whether a specific bbox+dim combination is already cached (avoids a redundant DEM fetch).

In [ ]:
resp = requests.get("http://127.0.0.1:9000/api/cache/check", params={
    **s.bbox,
    "dim": s.settings["dem"]["dim"],
})
resp.raise_for_status()
print(resp.json())

### Clear cache  `DELETE /api/cache`

In [ ]:
# Uncomment to wipe all cached files
# s.clear_cache()

---
## 8 — Full pipeline  `run_all()`

Convenience wrapper: `fetch_dem → show_dem → export_obj → verify → slice`.

In [ ]:
with TerrainSession() as s:
    s.start()
    s.select("WestAmerica")
    s.settings["dem"]["dim"]          = 800
    s.settings["split"]["split_rows"] = 4
    s.settings["split"]["split_cols"] = 4
    s.settings["slicer"]["slicer_config"] = "maps_2025_part2.ini"
    s.run_all()
# Server is stopped automatically on context manager exit

---
## 9 — Stop server

In [ ]:
s.stop()